# 機体点群の間引き
+ SCX900向けに衝突判定に用いられる機体点群を除去する
+ lightningを作るためのプログラム一覧

## ライブラリ

In [ ]:
import os

import numpy as np
import pandas as pd
import k3d

In [ ]:
from argus_synchro.experiments.debug_vis.viewer_3d import create_simple_k3d_points

## 定数

In [ ]:
# notebookの実行パスからプロジェクトのルートへの相対パス
path_to_root = "../"

In [ ]:
from configparser import ConfigParser, ExtendedInterpolation

from argus_synchro import shared_app_config
from argus_synchro.config.app_config import AppConfig
# import shared_app_config

In [ ]:
app_ini = ConfigParser(interpolation=ExtendedInterpolation())
app_ini.read(f"{path_to_root}/config/settings.ini", "UTF-8")
# 共有メモリに反映
app_config = AppConfig(app_ini)
# sac = shared_app_config.SharedAppConfig()
# app_config = sac.read()

## 機体関連の情報を読み込み

In [ ]:
import argus_synchro.SubScrutinizer as SubScrt
from argus_synchro.config.machine_collision import load_machine_info
from argus_synchro.experiments import py_machine_info_to_cpp

# import SubScrutinizer as SubScrt
# from config.machine_collision import load_machine_info
# from experiments import py_machine_info_to_cpp

In [ ]:
machine_dir = f"{path_to_root}/config/crane3d/collision_detection/SCX900-3/weighted"
json_machine_info = f"{path_to_root}/config/crane3d/collision_detection/SCX900-3/weighted/col_machine_info.jsonc"

In [ ]:
(l_machine_col_weighted, _,_ ) = SubScrt.create_machine_points(
    machine_dir,
    app_config.LiDARPosition,
    l_col_machine_conf = py_machine_info_to_cpp(load_machine_info(json_machine_info)),
)

In [ ]:
l_machine_points = [
    machine_col_parts.machine_pcd_points for machine_col_parts in l_machine_col_weighted
]

# 上部旋回体だけ範囲除去

In [ ]:
upper_parts_ind = 0
upper_machine_points = l_machine_points[upper_parts_ind]

In [ ]:
center_points = upper_machine_points.mean(axis=0)
center_points[0] -= 0.7
center_points[2] = 0

proc_upper_machine_points = upper_machine_points - center_points
remove_ind = (
    np.abs(upper_machine_points - center_points) <= np.array([1.5, 1.2, 1000])
).all(axis=1)
l_machine_points[upper_parts_ind] = upper_machine_points[~remove_ind]

In [ ]:
plot = k3d.plot()

plot += create_simple_k3d_points(
    l_machine_points[upper_parts_ind],
    color=0x0000ff,
    point_size=0.05,
)

plot.display()

# 各機体部品を高さ方向に輪切りにする

In [ ]:
cut_points = [0.2, 0.4, 0.6, 0.8]

In [ ]:
pick_z_indices = []
for machine_points in l_machine_points:
    chosen_points = np.quantile(machine_points[:, 2], q=cut_points)
    pick_z_indices.append(
        np.vstack(
            [
                (np.abs(machine_points[:, 2] - chosen_points) < 0.01)
                for chosen_points in chosen_points
            ]
        ).any(axis=0)
    )
    pass

In [ ]:
pick_machine_points = [
    machine_points[pick_z_ind]
    for pick_z_ind, machine_points in zip(pick_z_indices, l_machine_points)
]

In [ ]:
[
    len(elem)
    for elem in pick_machine_points
]

In [ ]:
plot = k3d.plot()
plot += create_simple_k3d_points(
    np.vstack(pick_machine_points),   
)
plot.display()

## 書き込み

In [ ]:
export_dir = f"{path_to_root}/config/crane3d/collision_detection/SCX900-3/test_lightning"
os.makedirs(export_dir, exist_ok=True)

In [ ]:
machine_dir = f"{path_to_root}/config/crane3d/collision_detection/SCX900-3/weighted"

In [ ]:
for picke_machine_parts_points, machine_col_parts in zip(
    pick_machine_points, l_machine_col_weighted
):
    filename = (
        export_dir
        + "/"
        + machine_col_parts.pcd_points_file
    )
    np.savetxt(
        filename,
        picke_machine_parts_points,
    )